In [ ]:
%pip install -q psycopg2-binary pymongo dnspython certifi python-dotenv pandas

In [ ]:
# Import the necessary libraries
import os

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from pymongo import MongoClient
from psycopg2 import Error
from psycopg2.extras import execute_values

load_dotenv(dotenv_path=".env", override=True)

True

In [ ]:
# PostgreSQL connection details from .env
hostname = os.getenv("hostnamePostgreSQL")
database = os.getenv("databasePostgreSQL")
port = os.getenv("portPostgreSQL")
username = os.getenv("usernamePostgreSQL")
password = os.getenv("passwordPostgreSQL")

connection = None
try:
    connection = psycopg2.connect(
        host=hostname,
        database=database,
        user=username,
        password=password,
        port=port,
    )
    cursor = connection.cursor()
    cursor.execute("SELECT version();")
    print("Connected to:", cursor.fetchone()[0])
except Error as error:
    print("Error while connecting to PostgreSQL:", error)
finally:
    if connection is not None:
        cursor.close()
        connection.close()
        print("PostgreSQL connection is closed")

In [ ]:
# Preview the payment dataset
order_payments = pd.read_csv("data/olist_order_payments_dataset.csv")
order_payments.head()

In [ ]:
# Upload the payment dataset to PostgreSQL
csv_file_path = "data/olist_order_payments_dataset.csv"
table_name = "olist_order_payments"
connection = None

try:
    connection = psycopg2.connect(
        host=hostname,
        database=database,
        user=username,
        password=password,
        port=port,
    )
    cursor = connection.cursor()
    print("Connected to PostgreSQL successfully!")

    cursor.execute(f"DROP TABLE IF EXISTS {table_name};")
    cursor.execute(
        f"""
        CREATE TABLE {table_name} (
            order_id VARCHAR(50),
            payment_sequential INTEGER,
            payment_type VARCHAR(20),
            payment_installments INTEGER,
            payment_value NUMERIC(10, 2)
        );
        """
    )

    data = pd.read_csv(csv_file_path)
    batch_size = 500
    total_records = len(data)

    insert_query = f"""
        INSERT INTO {table_name}
        (order_id, payment_sequential, payment_type, payment_installments, payment_value)
        VALUES %s;
    """

    for start in range(0, total_records, batch_size):
        end = min(start + batch_size, total_records)
        batch_records = list(data.iloc[start:end].itertuples(index=False, name=None))
        execute_values(cursor, insert_query, batch_records)
        print(f"Inserted records {start + 1} to {end}")

    connection.commit()
    print(f"All {total_records} records inserted into {table_name}.")

except Error as error:
    if connection is not None:
        connection.rollback()
    print("Error while loading data into PostgreSQL:", error)
finally:
    if connection is not None:
        cursor.close()
        connection.close()
        print("PostgreSQL connection is closed")

In [21]:
# importing module
from pymongo import MongoClient

hostname = os.getenv("hostnameMongoDB")
database = os.getenv("databaseMongoDB")
port = os.getenv("portMongoDB")
username = os.getenv("usernameMongoDB")
password = os.getenv("passwordMongoDB")

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]


In [22]:
# Upload product-category translations to MongoDB
product_category_df = pd.read_csv(
    "data/product_category_name_translation.csv",
    encoding="utf-8-sig",
)

try:
    collection = mydatabase["product_categories"]
    records = product_category_df.to_dict(orient="records")
    result = collection.insert_many(records)
    print(f"Uploaded {len(result.inserted_ids)} records to MongoDB successfully!")
except Exception as error:
    print(f"An error occurred: {error}")
finally:
    client.close()
    print("MongoDB connection is closed")

Uploaded 71 records to MongoDB successfully!
MongoDB connection is closed
